# Q-Learning

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/02-q-learning

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Tabular Q-learning, from scratch

Model-free: the agent never sees the transition rules — it learns $Q(s,a)$ purely from sampled experience using the temporal-difference update.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)

## The TD update + ε-greedy exploration

$Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big]$, behaving with a decaying ε-greedy policy.

In [ ]:
def q_learning(env, episodes=2000, alpha=0.5, gamma=0.9):
    rng = np.random.RandomState(0)
    Q = np.zeros((env.nS, env.nA))
    eps, returns = 1.0, []
    for ep in range(episodes):
        s, done, total, steps = 0, False, 0, 0
        while not done and steps < 100:
            a = rng.randint(env.nA) if rng.rand() < eps else int(np.argmax(Q[s]))
            s2, r, done = env.step(s, a)
            Q[s,a] += alpha * (r + gamma*np.max(Q[s2]) - Q[s,a])   # off-policy TD
            s = s2; total += r; steps += 1
        eps = max(0.05, eps*0.999); returns.append(total)
    return Q, returns

Q, returns = q_learning(env)
print('learned. final greedy policy reaches goal from start.')

## Learning curve and learned policy

In [ ]:
import numpy as np
ma = np.convolve(returns, np.ones(50)/50, mode='valid')
fig, ax = plt.subplots(1, 2, figsize=(13,5))
ax[0].plot(ma, color='#14b8a6'); ax[0].set_xlabel('episode'); ax[0].set_ylabel('return (50-ep avg)')
ax[0].set_title('Q-learning improves with experience')
arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = np.argmax(Q, axis=1)
ax[1].imshow(Q.max(1).reshape(5,5), cmap='viridis')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax[1].text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax[1].set_title('max Q (color) + greedy policy'); ax[1].axis('off'); plt.show()

## Off-policy in action

The update used $\max_{a'}Q(s',a')$ — the *best* next action — even though the agent often explored randomly. That is why Q-learning converges to the **optimal** policy while behaving sub-optimally.

In [ ]:
# Greedy rollout from the start with the learned Q
s, path, done = 0, [0], False
while not done and len(path) < 20:
    s, _, done = env.step(s, int(np.argmax(Q[s]))); path.append(s)
print('greedy path (state ids):', path)
print('reached goal:', path[-1] == env.goal, 'in', len(path)-1, 'steps (optimal = 8)')

## Key takeaways

- Q-learning learns $Q^*$ from sampled `(s,a,r,s')` with no model of the environment.
- The **TD update** bootstraps from the agent's own next estimate.
- It is **off-policy**: the target uses the best next action regardless of behavior.
- **ε-greedy** with decay balances exploration early and exploitation later.